In [1]:
import random
import heapq

GRID = 6
start = (0,0)
goal = (5,5)

MOVE_COST = 1
HOVER_COST = 0.5
REPLAN_COST = 3

battery_adapt = 100
battery_replan = 100

print("=== Adaptation vs Replanning Utility Demo ===")

# ------------------------------------------------
# create grid
# ------------------------------------------------

def create_grid(obstacles):

    grid=[["." for _ in range(GRID)] for _ in range(GRID)]

    grid[start[0]][start[1]]="S"
    grid[goal[0]][goal[1]]="G"

    for o in obstacles:
        grid[o[0]][o[1]]="X"

    return grid


def show_grid(grid):

    for r in grid:
        print(r)

# ------------------------------------------------
# base planned path
# ------------------------------------------------

def planned_path():

    path=[start]
    r,c=start

    while (r,c)!=goal:

        if c<goal[1]:
            c+=1
        elif r<goal[0]:
            r+=1

        path.append((r,c))

    return path


base_plan=planned_path()

# ------------------------------------------------
# generate obstacles ON PATH
# ------------------------------------------------

obstacles=random.sample(base_plan[2:-1],4)

# create separate obstacle environments
obstacles_adapt=obstacles.copy()
obstacles_replan=obstacles.copy()

grid=create_grid(obstacles)

print("\nEnvironment Grid\n")
show_grid(grid)

print("\nBase Planned Path:",base_plan)
print("Obstacles:",obstacles)

# ------------------------------------------------
# A* replanning
# ------------------------------------------------

def neighbors(node):

    r,c=node
    dirs=[(1,0),(-1,0),(0,1),(0,-1)]

    valid=[]

    for dr,dc in dirs:

        nr=r+dr
        nc=c+dc

        if 0<=nr<GRID and 0<=nc<GRID:
            valid.append((nr,nc))

    return valid


def heuristic(a,b):

    return abs(a[0]-b[0])+abs(a[1]-b[1])


def astar(start,goal,obstacles):

    pq=[]
    heapq.heappush(pq,(0,start))

    came={}
    cost={start:0}

    while pq:

        _,node=heapq.heappop(pq)

        if node==goal:
            break

        for n in neighbors(node):

            if n in obstacles:
                continue

            new_cost=cost[node]+1

            if n not in cost or new_cost<cost[n]:

                cost[n]=new_cost
                priority=new_cost+heuristic(n,goal)

                heapq.heappush(pq,(priority,n))
                came[n]=node

    path=[goal]

    while path[-1]!=start:
        path.append(came[path[-1]])

    path.reverse()

    return path


# ------------------------------------------------
# ADAPTATION EXECUTION
# ------------------------------------------------

print("\n===== ADAPTATION STRATEGY =====")

position=start

for step in base_plan[1:]:

    print("\nDrone attempting:",step)

    if step in obstacles_adapt:

        print("⚠ STATE MONITORING: obstacle detected")

        print("Adaptive strategy → hover and wait")

        battery_adapt-=HOVER_COST

        print("Obstacle clears at t+1")

        obstacles_adapt.remove(step)

    position=step

    battery_adapt-=MOVE_COST

    print("Drone moved to",position)
    print("Battery remaining:",battery_adapt)

    if position==goal:
        break


# ------------------------------------------------
# REPLANNING EXECUTION
# ------------------------------------------------

print("\n===== REPLANNING STRATEGY =====")

position=start
plan=base_plan.copy()

while position!=goal:

    next_step=None

    for s in plan:
        if s==position:
            idx=plan.index(s)
            next_step=plan[idx+1]
            break

    print("\nDrone attempting:",next_step)

    if next_step in obstacles_replan:

        print("⚠ PLAN MONITORING: path blocked")

        print("Replanning new route")

        battery_replan-=REPLAN_COST

        new_path=astar(position,goal,obstacles_replan)

        print("New Path:",new_path)

        plan=new_path

        continue

    position=next_step

    battery_replan-=MOVE_COST

    print("Drone moved to",position)
    print("Battery remaining:",battery_replan)


# ------------------------------------------------
# UTILITY COMPARISON
# ------------------------------------------------

print("\n===== UTILITY COMPARISON =====")

print("Battery remaining after ADAPTATION:",battery_adapt)
print("Battery remaining after REPLANNING:",battery_replan)

if battery_adapt>battery_replan:
    print("\nAdaptation has better utility")
else:
    print("\nReplanning has better utility")

=== Adaptation vs Replanning Utility Demo ===

Environment Grid

['S', '.', 'X', '.', 'X', 'X']
['.', '.', '.', '.', '.', 'X']
['.', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', '.']
['.', '.', '.', '.', '.', 'G']

Base Planned Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5)]
Obstacles: [(1, 5), (0, 2), (0, 4), (0, 5)]

===== ADAPTATION STRATEGY =====

Drone attempting: (0, 1)
Drone moved to (0, 1)
Battery remaining: 99

Drone attempting: (0, 2)
⚠ STATE MONITORING: obstacle detected
Adaptive strategy → hover and wait
Obstacle clears at t+1
Drone moved to (0, 2)
Battery remaining: 97.5

Drone attempting: (0, 3)
Drone moved to (0, 3)
Battery remaining: 96.5

Drone attempting: (0, 4)
⚠ STATE MONITORING: obstacle detected
Adaptive strategy → hover and wait
Obstacle clears at t+1
Drone moved to (0, 4)
Battery remaining: 95.0

Drone attempting: (0, 5)
⚠ STATE MONITORING: obstacle detected
Adaptive strategy → h